# Конспект. Модуль 6: Контентная фильтрация (Content-Based)

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 6 из 13 — «Контентная фильтрация (Content-Based)»
**Цель модуля:** освоить подход, принципиально отличный от всего, что было в Модулях 3–5, — рекомендацию **без использования истории взаимодействий других людей**. Это осознанный шаг назад по сложности математики (после плотной линейной алгебры Модуля 5) и одновременно решение самой острой нерешённой проблемы предыдущих модулей: что делать, если у товара вообще нет ни одного взаимодействия.

**Связь с предыдущими модулями:** все алгоритмы Модулей 3–5 (UB-CF, IB-CF, Funk SVD, ALS) **принципиально не способны** дать никакого предсказания для товара, у которого нет ни единой известной оценки `r_ui` — им попросту не из чего строить сходство или обучать латентный вектор `q_i`. Контентная фильтрация — единственный подход из рассмотренных до сих пор, который не имеет этого ограничения.

## 6.1 Идея

### 6.1.1 Основной принцип

Если в Модулях 3–4 источником сигнала было «кто ещё это оценивал», а в Модуле 5 — «скрытая структура, обученная на всей матрице взаимодействий», то здесь источник сигнала — **сам товар**, его объективные характеристики. Формулировка: «рекомендуем товары, похожие на те, что пользователь уже полюбил, — похожие по содержанию, а не по тому, кто их ещё оценивал».

### 6.1.2 Формальное отличие от коллаборативной фильтрации

| | Коллаборативная фильтрация (Модули 3–5) | Контентная фильтрация (этот модуль) |
|:---|:---|:---|
| Источник сигнала | Матрица взаимодействий `других` пользователей | Метаданные/содержание самого товара |
| Нужна ли история взаимодействий с товаром | Да, критично | Нет — достаточно описания товара |
| Единица сравнения | Пользователи или товары **через** общих пользователей | Товары **напрямую**, по признакам |
| Что если товар совсем новый | Не работает (Модуль 1.5.1, cold start товара) | Работает с первого дня |

### 6.1.3 Ключевое преимущество — прямое решение cold start товара

Вернёмся к формуле Funk SVD (Модуль 5.4.1): `L = Σ_(u,i)∈known (...)`. Если для товара `i` множество `known` пусто (ни одной оценки), слагаемое для этого товара просто отсутствует в функции потерь — вектор `q_i` остаётся необученным, случайным. Контентная фильтрация не имеет этой зависимости: как только у нового товара появилось **описание** (жанр, категория, текст), можно немедленно построить его профиль (раздел 6.2) и включить в рекомендации — не дожидаясь, пока накопится статистика взаимодействий.

### 6.1.4 Ограничение — низкая serendipity (прямая связь с Модулем 1.5.5)

Плата за это преимущество: контентная фильтрация структурно **не способна** найти неочевидные связи, не отражённые в признаках товара. Она порекомендует «ещё один фильм про космос» тому, кто смотрел фильм про космос — предсказуемо, но скучно (низкая **Novelty** и **Serendipity**, Модуль 1.5.5). Найти неожиданную, но точную рекомендацию (например, «фильм с совершенно другим сюжетом, но с той же эмоциональной динамикой, которую вы явно любите») контентная фильтрация не может в принципе, потому что она видит только явные признаки, а не скрытые паттерны совместного потребления, которые улавливает коллаборативная фильтрация. Это прямая мотивация Модуля 7 (гибридные системы) — совместить сильные стороны обоих подходов.

## 6.2 Профиль товара (Item Profile)

### 6.2.1 Типы признаков

Профиль товара — это, по сути, тот же самый процесс feature engineering, с которым вы уже детально работали (Неделя 5 общего плана, `sklearn.compose.ColumnTransformer`) — только признаки описывают не транзакцию, а сам товар:

| Тип признака | Примеры | Как векторизовать |
|:---|:---|:---|
| Текстовые | Название, описание, жанры, теги | TF-IDF (раздел 6.3) |
| Числовые | Год выпуска, цена, рейтинг критиков | Масштабирование (`StandardScaler`, уже знакомо) |
| Категориальные | Категория, бренд, режиссёр, исполнитель | One-Hot Encoding (уже знакомо) |

### 6.2.2 Мультимодальный профиль — то же самое, что вы уже делали в FraudGuard/ML Pipeline

Полный профиль товара на практике — это конкатенация всех этих типов признаков в единый вектор, собранный ровно тем же архитектурным паттерном `ColumnTransformer`, который вы уже применяли:

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

item_profile_pipeline = ColumnTransformer([
    ('text', TfidfVectorizer(), 'description'),           # текстовое описание
    ('numeric', StandardScaler(), ['release_year', 'price']),  # числовые признаки
    ('categorical', OneHotEncoder(handle_unknown='ignore'), ['category', 'brand']),  # категориальные
])

item_vectors = item_profile_pipeline.fit_transform(items_df)

**Важное отличие от вашего опыта с FraudGuard:** там `ColumnTransformer` строил признаки **транзакции** для предсказания вероятности фрода. Здесь та же самая техническая конструкция строит признаки **товара** для последующего сравнения товаров между собой через косинусное сходство (Модуль 2.2.1) — суть инструмента не изменилась, изменился только объект, который мы описываем.

## 6.3 TF-IDF: от текста к вектору

Текстовые/категориальные признаки (жанры, теги, описание) — обычно самая информативная часть профиля товара, поэтому им уделяется отдельное внимание.

### 6.3.1 Term Frequency (TF)

Как часто термин (слово, жанр, тег) встречается в описании конкретного товара. В простейшем варианте — просто счётчик присутствия: `TF=1`, если жанр указан, `TF=0`, если нет (именно этот вариант используется в примере ниже; существуют и более сложные варианты — нормализация по длине описания, логарифмическое сглаживание, — но для коротких структурированных полей вроде списка жанров простого бинарного счётчика обычно достаточно).

### 6.3.2 Inverse Document Frequency (IDF)

In [ ]:
IDF(term) = ln(N / df(term))

где `N` — общее число товаров («документов»), `df(term)` — число товаров, содержащих этот термин. **Смысл:** редкие термины несут больше различающей информации, чем частые. Термин, который есть почти у всех товаров («фильм», «товар»), почти бесполезен для различения — его `IDF` близок к нулю (при `df->N`, `IDF->ln(1)=0`). Редкий термин, встречающийся у 1 товара из 1000, наоборот, получает высокий вес.

**TF-IDF = TF × IDF** — итоговый вес термина в профиле конкретного товара.

### 6.3.3 Полный проверенный численный пример — продолжение сквозного примера курса

Присвоим товарам `I1`–`I5` из уже знакомой вам матрицы (Модули 3–5) жанровые теги. Выбор жанров не случаен — он согласован с уже обнаруженными в данных паттернами сходства: напомним, в Модуле 4.3.3 мы установили, что **I1 и I4 получили абсолютно идентичные оценки** от всех пользователей, оценивших оба товара (`sim(I4,I1)=1.0`) — это сильный сигнал, что I1 и I4 по сути «один и тот же вид» контента. Присвоим им одинаковый жанр:

| Товар | Жанры |
|:---|:---|
| I1 | Action, SciFi |
| I2 | Romance, Drama |
| I3 | Action, Drama |
| I4 | Action, SciFi *(идентично I1)* |
| I5 | Romance, Comedy |

**Document Frequency (`N=5` товаров):**

In [ ]:
Action:  3 (I1, I3, I4)
SciFi:   2 (I1, I4)
Romance: 2 (I2, I5)
Drama:   2 (I2, I3)
Comedy:  1 (I5)

**IDF (проверено вычислением):**

In [ ]:
IDF(Action)  = ln(5/3) = 0.5108
IDF(SciFi)   = ln(5/2) = 0.9163
IDF(Romance) = ln(5/2) = 0.9163
IDF(Drama)   = ln(5/2) = 0.9163
IDF(Comedy)  = ln(5/1) = 1.6094

**Итоговые TF-IDF векторы** (порядок компонент: `[Action, SciFi, Romance, Drama, Comedy]`):

In [ ]:
I1 = [0.5108, 0.9163, 0,      0,      0     ]
I2 = [0,      0,      0.9163, 0.9163, 0     ]
I3 = [0.5108, 0,      0,      0.9163, 0     ]
I4 = [0.5108, 0.9163, 0,      0,      0     ]   <- идентичен I1, как и ожидалось
I5 = [0,      0,      0.9163, 0,      1.6094]

### 6.3.4 Сглаженная версия IDF (как в `sklearn`)

Формула `ln(N/df)` из 6.3.2 — учебная, упрощённая версия. Проблема: если термин встречается **во всех** документах (`df=N`), `IDF=ln(1)=0` — термин полностью исчезает из представления, что иногда нежелательно. `sklearn.feature_extraction.text.TfidfVectorizer` по умолчанию использует сглаженную версию:

In [ ]:
IDF_smooth(term) = ln((1+N) / (1+df(term))) + 1

Дополнительно `sklearn` по умолчанию **нормализует каждый вектор документа по L2-норме** (проверено: для `I1` сумма квадратов компонент результата `sklearn` равна ровно `1.0`). Для целей курса и понимания механики мы используем простую версию (6.3.2) — она даёт идентичные по смыслу, хотя и не идентичные по абсолютным числам результаты, — но при использовании готовой библиотеки в реальном проекте (раздел 6.5) стоит знать, что абсолютные значения будут немного другими из-за этого сглаживания и нормализации.

## 6.4 Профиль пользователя (User Profile)

### 6.4.1 Формула

Профиль пользователя строится как **взвешенное среднее** TF-IDF векторов товаров, которые он оценил, с весами, равными выставленным оценкам (или, в implicit-варианте, просто равными весами для всех товаров, с которыми было взаимодействие):

In [ ]:
profile_u = Σ_i∈rated(u) (r_ui × vector_i) / Σ_i∈rated(u) r_ui

### 6.4.2 Полный проверенный численный пример — построение профиля U1

Используем уже известные оценки U1 из сквозной матрицы курса: `I1=5, I2=3, I3=4`.

In [ ]:
profile_U1 = (5×I1_vec + 3×I2_vec + 4×I3_vec) / (5+3+4)

5×I1 = [2.554,  4.5815, 0,      0,      0]
3×I2 = [0,      0,      2.7489, 2.7489, 0]
4×I3 = [2.0432, 0,      0,      3.6652, 0]

сумма = [4.5972, 4.5815, 2.7489, 6.4141, 0]

profile_U1 = сумма / 12 = [0.3831, 0.3818, 0.2291, 0.5345, 0]

**Рекомендация — косинусное сходство профиля с ещё не оценёнными товарами I4 и I5** (Модуль 2.2.1):

In [ ]:
cos(profile_U1, I4) = 0.6548
cos(profile_U1, I5) = 0.1427

**Интерпретация:** контентная модель уверенно рекомендует **I4** (высокое сходство — потому что I4 разделяет с I1 жанры Action и SciFi, а I1 U1 оценил максимально высоко) и значительно менее уверена в I5 (низкое сходство — I5 относится к жанрам Romance/Comedy, слабо пересекающимся с тем, что нравится U1, единственная связь — через умеренную оценку I2, где есть общий жанр Drama лишь частично, а Romance у U1 в профиле присутствует слабо).

### 6.4.3 Итоговое сравнение всех методов курса на одной и той же задаче

Мы предсказывали рейтинг/предпочтение U1 к I4 (относительно I5) уже шестью независимыми методами на протяжении курса. Сведём всё воедино:

| Метод | Модуль | Результат для U1 |
|:---|:---:|:---|
| User-Based CF | 3 | `pred(U1,I4) = 5.000` (после клиппинга) |
| Item-Based CF (только положительные соседи) | 4 | `pred(U1,I4) = 4.566` |
| Funk SVD | 5.4 | `pred(U1,I4) = 4.939` |
| Explicit ALS | 5.5 | `pred(U1,I4) = 4.669` |
| Implicit ALS (ранжирование) | 5.5.5 | `I4 (0.826) > I5 (0.317)` |
| Content-Based | 6.4.2 | `cos(U1,I4)=0.655 > cos(U1,I5)=0.143` |

**Все шесть принципиально разных алгоритмов** — использующих совершенно разные источники сигнала (сходство пользователей, сходство товаров через рейтинги, обученные скрытые факторы, явный градиентный спуск против закрытого решения, implicit-переформулировка, и, наконец, чистое содержание товара без единого взгляда на матрицу оценок) — **согласованно** приходят к одному и тому же практическому выводу: U1, скорее всего, высоко оценит I4. Это самый весомый практический аргумент во всём модуле — когда независимые по своей математической природе методы сходятся в выводе, это значительно более сильное свидетельство, чем результат любого одного из них по отдельности. Именно на этом принципе (согласованность нескольких независимых сигналов) строятся гибридные системы, к которым мы переходим в Модуле 7.

### 6.4.4 Альтернативные схемы взвешивания профиля

- **Implicit-вариант (без явных рейтингов):** все взаимодействия получают одинаковый вес `1` — профиль становится простым (невзвешенным) средним TF-IDF векторов всех товаров, с которыми пользователь взаимодействовал.
- **Учёт негативной обратной связи:** если доступны явные дизлайки/низкие оценки, можно строить профиль как разность «среднего вектора понравившегося» и «среднего вектора не понравившегося» — это позволяет модели не просто тянуться к похожему, но и явно отталкиваться от нелюбимых характеристик.
- **Затухание по времени (time decay):** более свежие взаимодействия получают больший вес, чем старые — прямая связь с уже обсуждавшейся нестабильностью вкусов во времени (Модуль 3.3.2).

## 6.5 Практика

### 6.5.1 Класс `ContentBasedRecommender`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class ContentBasedRecommender:
    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.item_vectors = None
        self.item_ids = None

    def fit(self, items_df: pd.DataFrame, text_column: str = 'genres'):
        """items_df: DataFrame с колонками item_id и text_column (например, жанры через пробел)."""
        self.item_ids = items_df['item_id'].values
        self.item_vectors = self.vectorizer.fit_transform(items_df[text_column])

    def recommend(self, user_ratings: dict, k: int = 5) -> list:
        """
        user_ratings: {item_id: rating} - оценки пользователя.
        Возвращает top-k item_id, не входящих в user_ratings, отсортированных по сходству.
        """
        rated_mask = np.isin(self.item_ids, list(user_ratings.keys()))
        rated_indices = np.where(rated_mask)[0]
        weights = np.array([user_ratings[self.item_ids[idx]] for idx in rated_indices])

        # Взвешенный профиль пользователя (формула 6.4.1)
        rated_vectors = self.item_vectors[rated_indices]
        profile = np.asarray(rated_vectors.T @ weights / weights.sum()).flatten()

        # Сходство профиля со всеми товарами
        similarities = cosine_similarity(profile.reshape(1, -1), self.item_vectors).flatten()

        # Исключаем уже оценённые товары и берём top-k
        similarities[rated_mask] = -1
        top_k_idx = np.argsort(-similarities)[:k]

        return [(self.item_ids[idx], similarities[idx]) for idx in top_k_idx]

# Воспроизведение примера из 6.3.3-6.4.2
items = pd.DataFrame({
    'item_id': ['I1','I2','I3','I4','I5'],
    'genres': ['Action SciFi', 'Romance Drama', 'Action Drama', 'Action SciFi', 'Romance Comedy']
})

recommender = ContentBasedRecommender()
recommender.fit(items)
recommendations = recommender.recommend({'I1': 5, 'I2': 3, 'I3': 4}, k=2)
print(recommendations)  # ожидаем I4 с более высоким сходством, чем I5

### 6.5.2 Мультимодальный профиль на MovieLens (жанры + год выпуска)

- Загрузить `movies.dat` из MovieLens 1M (колонки: `movie_id`, `title`, `genres` — жанры разделены `|`).
- Извлечь год выпуска из title (регулярное выражение на `(\d{4})`).
- Построить `ColumnTransformer` (раздел 6.2.2): `TfidfVectorizer` на жанрах + `StandardScaler` на годе выпуска.
- Реализовать `ContentBasedRecommender`, использующий этот объединённый профиль, и получить топ-10 рекомендаций для нескольких реальных пользователей.

### 6.5.3 Сравнение с Item-Based CF (Модуль 4) — практическое задание

- Для одного и того же пользователя получить топ-5 рекомендаций от `ContentBasedRecommender` (этот модуль) и от реализации Item-Based CF (Модуль 4.5.1).
- Посчитать пересечение (`Jaccard`, Модуль 2.2.4, — забавное возвращение метрики из совсем другого контекста!) между двумя списками рекомендаций.
- **Обсуждение:** для фильмов с очень характерным, узким жанром (например, нишевое артхаус-кино) контентная и коллаборативная модели обычно совпадают сильнее — почему? Для фильмов с широким, «размытым» жанровым профилем (например, комедийная драма) — обычно расходятся сильнее — почему?

### 6.5.4 Вопросы для самопроверки

1. Почему для товара с абсолютно новым, никогда раньше не встречавшимся жанром (`df=1` для этого жанра сразу после добавления) IDF будет **максимальным** среди всех терминов в словаре — и почему это концептуально правильно с точки зрения задачи рекомендаций?
2. В разделе 6.4.2 в профиле U1 компонента `Comedy` равна нулю. Объясните, почему — и что должно произойти (какое взаимодействие пользователя), чтобы эта компонента стала ненулевой?
3. Приведите пример ситуации (свой, не из конспекта), где контентная и коллаборативная модели дали бы **противоположные** рекомендации для одного и того же пользователя. Чем бы вы объяснили это расхождение?
4. Почему схема с затуханием по времени (6.4.4) особенно важна именно для контентной фильтрации, если вспомнить проблему нестабильности вкусов, впервые поднятую в Модуле 3.3.2 применительно к другому алгоритму?

## Глоссарий модуля 6

| Термин | Короткое определение |
|:---|:---|
| Content-Based Filtering | Рекомендация на основе схожести содержания товаров, без данных о других пользователях |
| Item Profile | Векторное представление товара по его признакам (текст, числа, категории) |
| TF (Term Frequency) | Частота/наличие термина в описании конкретного товара |
| IDF (Inverse Document Frequency) | Мера редкости термина по всему каталогу — редкие термины важнее |
| TF-IDF | Итоговый вес термина = TF × IDF |
| User Profile (контентный) | Взвешенное среднее векторов товаров, понравившихся пользователю |
| Filter Bubble / Over-specialization | Склонность контентной модели рекомендовать слишком похожее, без разнообразия |

**Связь со следующим модулем:** мы теперь располагаем двумя принципиально разными семействами методов — коллаборативным (Модули 3–5, использует поведение множества людей) и контентным (этот модуль, использует свойства самого товара) — каждое со своими сильными и слабыми сторонами, зеркально дополняющими друг друга (сравните 6.1.3–6.1.4 с проблемами Модулей 3.3–4.2). Модуль 7 формализует, как эти два подхода комбинировать в единую гибридную систему, а не выбирать между ними «или-или».